# CQITE Validation Notebook
Validates CQITE against exact imaginary-time evolution.

In [3]:
import sys
sys.path.append("../")

In [4]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm
from qiskit.quantum_info import Statevector, SparsePauliOp
from qitp.engine.unitary_approximation import UnitaryApproximation
from qitp.engine.propagation import QITEPropagation
from qitp.engine.cqite_run import run_qite, prepare_state
from qitp.simulation.backend import SimulatorBackend

In [5]:
from qitp.ansatz.pauli_basis import (
    PauliBasis
)
from qitp.ansatz.commutator_basis import (
    CommutatorBasis
)

In [6]:
h1 = 1
h2 = 1

Jz = 1

H = SparsePauliOp(
    ["XI", "IX", "ZZ"],
    coeffs=[1.0, 1.0, 1.0]
)

Hmat=H.to_matrix()


In [7]:
Mz = SparsePauliOp(
    ["YY"],
    coeffs=[1.0]
)

Identity = SparsePauliOp(["II"],coeffs=[1.0])

In [8]:
def exact_ite_step(psi,dt):
    U=expm(-dt*Hmat)
    x=U@psi
    return x/np.linalg.norm(x)


In [9]:
seeds = [
    SparsePauliOp("ZI"),
    SparsePauliOp("IZ"),
    SparsePauliOp("XI"),
    SparsePauliOp("IX"),
]

In [10]:
#basis = PauliBasis(seeds)

basis = CommutatorBasis.from_seed(hamiltonian=H, seeds=seeds, depth=1)

In [11]:
basis.size

8

In [12]:
approx = UnitaryApproximation(basis)

backend = SimulatorBackend()

prop = QITEPropagation(
    H,
    approx,
    backend
)

prop.constraints = [
    Identity,
    Mz
]


In [13]:
dt=1e-1
nsteps=20

energies=[]
fidelities=[]
errors=[]
magnetizations=[]

In [14]:
circuit = prepare_state("10")

In [15]:
results = run_qite(
    prop,
    circuit,
    H,
    Mz,
    dt,
    nsteps
)

energies = results["energies"]
magnetizations = results["magnetizations"]

In [31]:
print(np.round(energies[-1],3))

-1.963


In [ ]:
plt.plot(energies); plt.title('Energy'); plt.grid(); plt.show()

In [ ]:
plt.plot(magnetizations); plt.title("Magnetization"); plt.grid(); plt.show()

In [34]:
evals = np.linalg.eigvalsh(
    H.to_matrix()
)

print("Exact spectrum")

for e in evals:
    print(np.round(e,2))

Exact spectrum
-2.24
-1.0
1.0
2.24


In [36]:
import numpy as np
from scipy.optimize import root_scalar

Hm = H.to_matrix()
Mm = Mz.to_matrix()

target = 0.0


def mz_expectation(lam):

    Hc = Hm + lam * Mm

    E, V = np.linalg.eigh(Hc)

    psi = V[:, 0]

    return np.real(
        np.vdot(
            psi,
            Mm @ psi
        )
    )


sol = root_scalar(
    lambda lam:
        mz_expectation(lam) - target,
    bracket=[-100, 100]
)

lam = sol.root

Hc = Hm + lam * Mm

E, V = np.linalg.eigh(Hc)

psi_exact = V[:, 0]

E_exact = np.real(
    np.vdot(
        psi_exact,
        Hm @ psi_exact
    )
)

print("E_exact =", np.round(E_exact,2))

E_exact = -2.0
